In [ ]:
import numpy as np
import rasterio
import rioxarray
import matplotlib.pyplot as plt
import gc


def calculate_difference(interferogram1, interferogram2, chunk_size=1048):
    """
    Optimized difference calculation function with chunking validity checks and memory management
    
    Parameters:
    -----------
    interferogram1 : xarray.DataArray
        First interferogram data
    interferogram2 : xarray.DataArray
        Second interferogram data
    chunk_size : int
        Size of chunks for processing (default: 1048)
    
    Returns:
    --------
    numpy.ndarray
        Normalized difference array
    """
    # Use xarray objects to get unified shape information
    if interferogram1.shape != interferogram2.shape:
        raise ValueError("Input TIFF files must have the same dimensions")
    
    # Get data arrays and initialize output
    data1 = interferogram1.data
    data2 = interferogram2.data
    rows, cols = data1.shape
    difference = np.zeros((rows, cols), dtype=np.float32)
    
    # Optimize chunking strategy to prevent zero-size edge chunks
    chunk_rows = max(1, min(chunk_size, rows))
    chunk_cols = max(1, min(chunk_size, cols))
    
    for i in range(0, rows, chunk_rows):
        for j in range(0, cols, chunk_cols):
            # Calculate actual chunk bounds
            end_i = min(i + chunk_rows, rows)
            end_j = min(j + chunk_cols, cols)
            
            # Skip zero-size chunks
            if end_i <= i or end_j <= j:
                continue
            
            # Extract data chunks and validate
            chunk1 = data1[i:end_i, j:end_j]
            chunk2 = data2[i:end_i, j:end_j]
            
            # Calculate normalized difference with stability handling
            epsilon = 1e-8
            denominator = chunk1 + chunk2 + epsilon
            valid_mask = denominator != 0
            diff_chunk = np.full_like(chunk1, np.nan, dtype=np.float32)
            diff_chunk[valid_mask] = (chunk1[valid_mask] - chunk2[valid_mask]) / denominator[valid_mask]
            
            # Write results
            difference[i:end_i, j:end_j] = diff_chunk
    
    # Comprehensive mask processing
    mask = np.isnan(data1) | np.isnan(data2) | (data1 == 0) | (data2 == 0)
    difference[mask] = np.nan  # Set to NaN for consistent downstream processing
    
    return difference


def process_interferogram_difference(tif_path1, tif_path2, chunk_size=1048, show_plot=True, output_suffix="difference"):
    """
    Complete workflow for interferogram difference processing
    
    Parameters:
    -----------
    tif_path1 : str
        Path to first input TIFF file
    tif_path2 : str
        Path to second input TIFF file
    chunk_size : int
        Chunk size for calculation (default: 1048)
    show_plot : bool
        Whether to display visualization plots (default: True)
    output_suffix : str
        Output file suffix identifier (default: "difference")
    
    Returns:
    --------
    None
        Saves results as .npy and .tif files
    """
    # Load data
    ifg1 = rioxarray.open_rasterio(tif_path1)[0]
    ifg2 = rioxarray.open_rasterio(tif_path2)[0]
    
    # Calculate difference
    phase_diff = calculate_difference(ifg1, ifg2, chunk_size=chunk_size)
    
    # Mask processing
    phase_diff = np.where(np.isnan(ifg1), np.nan,
                         np.where(ifg1 == 0, 0, phase_diff))
    phase_diff = np.where(np.isnan(ifg2), np.nan,
                         np.where(ifg2 == 0, 0, phase_diff))
    
    # Save as numpy format
    np.save(tif_path2.replace(".tif", f"_{output_suffix}.npy"), phase_diff)
    
    # Read original geographic parameters
    with rasterio.open(tif_path2) as src:
        transform = src.transform
        width = src.width
        height = src.height
        dtype = src.dtypes[0]
        nodata = src.nodatavals[0]
        crs = src.crs
    
    # Save as GeoTIFF
    output_tif = tif_path2.replace(".tif", f"_{output_suffix}.tif")
    with rasterio.open(
        output_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=dtype,
        crs=crs,
        transform=transform,
        nodata=nodata,
    ) as dst:
        dst.write(phase_diff, 1)
    
    # Visualization
    if show_plot:
        plt.figure(figsize=(12, 6))
        plt.imshow(phase_diff, cmap="viridis", aspect="auto")
        plt.colorbar(label="Phase Difference")
        plt.title(f"Processed Result: {output_suffix}")
        plt.tight_layout()
        plt.show()
    
    # Memory cleanup
    del ifg1, ifg2, phase_diff
    gc.collect()
    
    print(f"Processing completed. Output saved as: {output_tif}")


# Example usage:
if __name__ == "__main__":
    # Replace with your actual file paths
    input_file1 = "/path/to/your/first/interferogram.tif"
    input_file2 = "/path/to/your/second/interferogram.tif"
    
    process_interferogram_difference(
        input_file1,
        input_file2,
        chunk_size=2048,
        show_plot=True,
        output_suffix="phase_difference"
    )

In [ ]:
import numpy as np
import rasterio
import rioxarray
import matplotlib.pyplot as plt
import gc


def process_norm_mask(tif_path, show_plot=True, output_suffix="norm_mask"):
    """
    Normal distribution threshold mask generation
    
    Parameters:
    tif_path (str): Input TIFF path
    show_plot (bool): Whether to display visualization
    output_suffix (str): Output file suffix
    """
    # Load data and get geographic parameters
    ifg = rioxarray.open_rasterio(tif_path)[0]
    delta_coh = ifg.values
    original_nodata = ifg.rio.nodata if ifg.rio.nodata is not None else 0
    
    # Get geographic parameters directly from rioxarray
    transform = ifg.rio.transform()
    crs = ifg.rio.crs
    height, width = ifg.rio.shape
    
    # Data preprocessing
    valid_mask = ~np.isnan(delta_coh) & (delta_coh != original_nodata)
    valid_values = delta_coh[valid_mask]  # Use 1D array directly
    
    # Normal distribution fitting to calculate threshold
    mean = np.mean(valid_values)
    std_dev = np.std(valid_values)
    threshold = mean+3*std_dev
    print(threshold)
    
    # Generate result matrix
    result = np.full_like(delta_coh, fill_value=original_nodata, dtype=delta_coh.dtype)
    result[(delta_coh >= threshold) & valid_mask] = delta_coh[(delta_coh >= threshold) & valid_mask]
    
    # Save GeoTIFF
    output_tif = tif_path.replace(".tif", f"_{output_suffix}.tif")
    with rasterio.open(
        output_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=result.dtype,
        crs=crs,
        transform=transform,
        nodata=original_nodata
    ) as dst:
        dst.write(result, 1)
    
    # Visualization
    if show_plot:
        plt.figure(figsize=(12, 6))
        plt.imshow(np.where(result == original_nodata, np.nan, result),
                  cmap="viridis", aspect="auto")
        plt.colorbar(label="Original Value")
        plt.title(f"Threshold: μ + 2σ = {threshold:.2f}")
        plt.show()
    
    # Memory cleanup
    del ifg, delta_coh, result
    gc.collect()


# Example usage:
if __name__ == "__main__":
    # Replace with your actual file path
    process_norm_mask(
        "/path/to/your/input_file.tif", 
        show_plot=True
    )